# Semantic PDF Q&A Pipeline (Local, Offline)
This version uses a unique hash-based FAISS index folder for each PDF to prevent overwriting.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
import hashlib, os

In [ ]:
pdf_path = 'data/Thermodynamics_by_PK_Nag.pdf'
embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
llm_model_name = 'mistral'

# Create a unique FAISS folder for this PDF
name_hash = hashlib.sha256(os.path.basename(pdf_path).encode()).hexdigest()[:12]
index_path = os.path.join('faiss_index', name_hash)
os.makedirs(index_path, exist_ok=True)

print(f'Using FAISS index folder: {index_path}')

In [ ]:
loader = PyMuPDFLoader(pdf_path)
docs = loader.load()
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
chunker = SemanticChunker(embeddings)
chunks = chunker.split_documents(docs)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(index_path)
retriever = vectorstore.as_retriever(search_kwargs={'k':3})
llm = ChatOllama(model=llm_model_name, temperature=0.3)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
print('Pipeline ready.')

In [ ]:
query = 'What is the first law of thermodynamics?'
print('Question:', query)
answer = qa_chain.run(query)
print('Answer:', answer)